In [ ]:
# Cell 1: imports and project-root setup
import sys
from pathlib import Path

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    candidates = [start] + list(start.parents)
    for candidate in candidates:
        if (candidate / "src").exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
import gym
import d4rl  # noqa: F401

from src.experiment_config import *
from src.config import CHECKPOINTS_DIR, RAW_METRICS_DIR, OBS_STATS_DIR
from src.dataset import NoisyOfflineRLDataset
from src.encoder import DisentangledEncoder
from src.riql import RIQLAgent
from src.train_eval import (
    eval_policy_on_env,
    load_and_freeze_encoder,
    save_metrics_json,
    train_riql_from_loader,
)

In [ ]:
# Distance correlation independence penalty
import torch

def _center_distance(D):
    row_mean   = D.mean(dim=1, keepdim=True)
    col_mean   = D.mean(dim=0, keepdim=True)
    total_mean = D.mean()
    return D - row_mean - col_mean + total_mean

def distance_correlation_loss(z1, z2):
    a = torch.cdist(z1, z1, p=2)
    b = torch.cdist(z2, z2, p=2)
    A = _center_distance(a)
    B = _center_distance(b)
    dcov2 = (A * B).mean()
    dvar1 = (A * A).mean()
    dvar2 = (B * B).mean()
    return dcov2 / torch.sqrt(dvar1 * dvar2 + 1e-12)

In [ ]:
# Cell 2: experiment configuration
# PPF-framework RIQL: DisentangledEncoder pretrained with privileged supervision
# and distance correlation independence penalty. RIQL used as the downstream algorithm.
METHOD = "disentangled_dcor_riql"

def noise_tag(noise_dim, noise_scale, noise_type):
    ns = str(noise_scale).replace(".", "p")
    return f"nd{noise_dim}_ns{ns}_{noise_type}"

NOISE_TAG = noise_tag(NOISE_DIM, NOISE_SCALE, NOISE_TYPE)
SEED_TAG  = f"seed_{SEED}"

CKPT_DIR    = CHECKPOINTS_DIR / METHOD / ENV_NAME / NOISE_TAG / SEED_TAG
METRICS_DIR = RAW_METRICS_DIR / METHOD / ENV_NAME / NOISE_TAG / SEED_TAG
OBS_DIR     = OBS_STATS_DIR   / METHOD / ENV_NAME / NOISE_TAG / SEED_TAG

for d in [CKPT_DIR, METRICS_DIR, OBS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DEVICE:", DEVICE)
print("METHOD:", METHOD)

In [ ]:
# Cell 3: dataset and dataloader
dataset = NoisyOfflineRLDataset(
    env_name=ENV_NAME,
    noise_dim=NOISE_DIM,
    noise_scale=NOISE_SCALE,
    seed=SEED,
    use_timeouts=True,
    noise_type=NOISE_TYPE,
)

train_loader = DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
    num_workers=4, pin_memory=True, persistent_workers=True,
)

state_dim      = dataset.noisy_obs.shape[1]
action_dim     = dataset.actions.shape[1]
true_state_dim = dataset.obs_dim
LATENT_DIM     = int(max(true_state_dim, NOISE_DIM) * 1.5)

np.savez(
    OBS_DIR / "obs_stats.npz",
    obs_mean=dataset.obs_mean,
    obs_std=dataset.obs_std,
    true_state_dim=true_state_dim,
)
print("state_dim (noisy):", state_dim)
print("true_state_dim:", true_state_dim)
print("action_dim:", action_dim)
print("latent_dim:", LATENT_DIM)

In [ ]:
# Cell 4: pretrain DisentangledEncoder with distance-correlation independence loss
# Identical to exp_disentangled_dcor.ipynb. Only the downstream algorithm differs.

torch.manual_seed(SEED)
np.random.seed(SEED)

encoder = DisentangledEncoder(
    state_dim=state_dim,
    action_dim=action_dim,
    true_state_dim=true_state_dim,
    latent_dim=LATENT_DIM,
).to(DEVICE)
optim = torch.optim.Adam(encoder.parameters(), lr=3e-4)

pretrain_loader = DataLoader(
    dataset, batch_size=PRETRAIN_BS, shuffle=True, drop_last=True,
    num_workers=4, pin_memory=True, persistent_workers=True,
)

for epoch in range(1, PRETRAIN_EPOCHS + 1):
    encoder.train()
    losses = []

    for obs, act, next_obs, rew, done, pure_obs, pure_next_obs in pretrain_loader:
        obs           = obs.to(DEVICE)
        act           = act.to(DEVICE)
        rew           = rew.to(DEVICE)
        pure_next_obs = pure_next_obs.to(DEVICE)

        z_task, z_irrel = encoder(obs)
        loss_dyn   = F.mse_loss(encoder.state_predictor(torch.cat([z_task, act], dim=-1)), pure_next_obs)
        loss_rew   = F.mse_loss(encoder.reward_predictor(z_task).squeeze(-1), rew)
        loss_indep = distance_correlation_loss(z_task, z_irrel)
        loss = loss_dyn + loss_rew + 0.05 * loss_indep

        optim.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(encoder.parameters(), max_norm=1.0)
        optim.step()
        losses.append(float(loss.item()))

    print(f"[pretrain] epoch {epoch}, loss={sum(losses)/len(losses):.4f}")

CKPT_ENCODER = CKPT_DIR / f"encoder_epoch_{PRETRAIN_EPOCHS}.pth"
torch.save(encoder.state_dict(), CKPT_ENCODER)
print("Saved encoder:", CKPT_ENCODER)

In [ ]:
# Cell 5: load and freeze encoder, then train RIQL
encoder = load_and_freeze_encoder(
    encoder=encoder,
    ckpt_path=CKPT_ENCODER,
    device=DEVICE,
)

riql = RIQLAgent(
    latent_dim=LATENT_DIM,
    action_dim=action_dim,
    device=DEVICE,
    expectile=0.7,
    temperature=3.0,
    discount=0.99,
    n_critics=10,
    quantile=0.25,
    huber_delta=1.0,
)

riql_history = train_riql_from_loader(
    riql=riql,
    train_loader=train_loader,
    device=DEVICE,
    epochs=EPOCHS,
    ckpt_dir=CKPT_DIR,
    method=METHOD,
    save_every=10,
    encoder=encoder,
    repr_mode="disentangled",
    use_tqdm=False,
)

In [ ]:
# Cell 6: evaluate and save metrics
print("Start evaluating ...")
metrics = eval_policy_on_env(
    iql=riql,
    env_name=ENV_NAME,
    encoder=encoder,
    method=METHOD,
    obs_mean=dataset.obs_mean,
    obs_std=dataset.obs_std,
    true_state_dim=true_state_dim,
    noise_dim=NOISE_DIM,
    noise_scale=NOISE_SCALE,
    noise_type=NOISE_TYPE,
    episodes=20,
    max_steps=1000,
    seed=SEED,
    device=DEVICE,
    use_fixed_noise=True,
)

metrics_path = save_metrics_json(
    metrics_dir=METRICS_DIR,
    metrics=metrics,
    env_name=ENV_NAME,
    method=METHOD,
    seed=SEED,
    noise_dim=NOISE_DIM,
    noise_scale=NOISE_SCALE,
    noise_type=NOISE_TYPE,
    extra={
        "latent_dim": LATENT_DIM,
        "pretrain_epochs": PRETRAIN_EPOCHS,
        "riql_epochs": EPOCHS,
    },
)
print(metrics)
print("Saved metrics:", metrics_path)